In [1]:
import os
import pandas as pd

# 1. ENSURE WE ARE AT ROOT (Run this again just in case)
if os.getcwd().endswith('analysis'):
    os.chdir('..')

# 2. DEFINE THE CORRECT NESTED PATH
# Based on your update, the data is inside src/ingest/
raw_data_path = os.path.join('src', 'ingest', 'data', 'raw')

print(f"Checking path: {os.path.abspath(raw_data_path)}")

# 3. VERIFY
if os.path.exists(raw_data_path):
    files = os.listdir(raw_data_path)
    print(f"✅ Found it! Files in raw: {files}")
else:
    print(f"❌ Still not found at {raw_data_path}")
    # Let's see what IS inside src/ingest just to be sure
    if os.path.exists('src/ingest'):
        print(f"Contents of src/ingest: {os.listdir('src/ingest')}")

Checking path: /workspaces/Climate-Data-Analysis-and-Prediciton-System/src/ingest/data/raw
✅ Found it! Files in raw: ['copernicus']


In [2]:
import xarray as xr
import os

# Navigate into the copernicus folder
copernicus_path = os.path.join('src', 'ingest', 'data', 'raw', 'copernicus')
files = [f for f in os.listdir(copernicus_path) if f.endswith(('.nc', '.grib'))]

if files:
    # Load the first file as a test
    test_file = os.path.join(copernicus_path, files[0])
    ds = xr.open_dataset(test_file)
    
    # Calculate missing values for a specific variable (e.g., temperature)
    # This is exactly what you need for the Milestone report!
    nan_summary = ds.isnull().sum().compute()
    print("--- Missing Value Summary ---")
    print(nan_summary)
else:
    print("📂 The copernicus folder is found, but it doesn't contain .nc or .grib files yet.")
    print(f"Current contents: {os.listdir(copernicus_path)}")

--- Missing Value Summary ---
<xarray.Dataset> Size: 16B
Dimensions:  ()
Coordinates:
    number   int64 8B 0
Data variables:
    t        int64 8B 0
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-22T07:30 GRIB to CDM+CF via cfgrib-0.9.1...


In [3]:
# 1. Check the physical size on your hard drive
file_size = os.path.getsize(test_file) / 1024  # Size in KB
print(f"File Size on Disk: {file_size:.2f} KB")

# 2. Look at the coordinates and dimensions
print("\n--- Dataset Dimensions ---")
print(ds.dims)

# 3. List all variables (Temperature is usually 't2m' or 'tp')
print("\n--- Data Variables ---")
print(ds.data_vars)

# 4. Check the actual values of 't'
print("\n--- Sample Values ---")
print(ds['t'].values)

File Size on Disk: 32965.40 KB

--- Dataset Dimensions ---
FrozenMappingWarningOnValuesAccess({'valid_time': 24, 'pressure_level': 1, 'latitude': 721, 'longitude': 1440})

--- Data Variables ---
Data variables:
    t        (valid_time, pressure_level, latitude, longitude) float32 100MB ...

--- Sample Values ---
[[[[251.08986 251.08986 251.08986 ... 251.08986 251.08986 251.08986]
   [251.16408 251.16408 251.16212 ... 251.16798 251.16603 251.16603]
   [251.56642 251.56447 251.56252 ... 251.57033 251.56837 251.56642]
   ...
   [285.42773 285.42578 285.42383 ... 285.4336  285.43164 285.4297 ]
   [285.58984 285.5879  285.5879  ... 285.5918  285.5918  285.58984]
   [286.04297 286.04297 286.04297 ... 286.04297 286.04297 286.04297]]]


 [[[250.4801  250.4801  250.4801  ... 250.4801  250.4801  250.4801 ]
   [250.443   250.443   250.443   ... 250.44495 250.44495 250.443  ]
   [250.34143 250.34143 250.34143 ... 250.34338 250.34143 250.34143]
   ...
   [286.53284 286.53088 286.52698 ... 286.5426

In [11]:
# Check for NaNs across the entire 100MB array
total_cells = ds.t.size
missing_values = ds.t.isnull().sum().values.item()
nan_percentage = (missing_values / total_cells) * 100

print(f"Total Data Points: {total_cells:,}")
print(f"Missing Values: {missing_values}")
print(f"NaN Rate: {nan_percentage:.4f}%")

# Quick check: Are there any infinite values?
inf_values = np.isinf(ds.t.values).sum()
print(f"Infinite Values: {inf_values}")

Total Data Points: 24,917,760
Missing Values: 0
NaN Rate: 0.0000%
Infinite Values: 0
